In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from collections import Counter

from sklearn.model_selection import train_test_split

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.13.0+cpu
Day 10 started.


In [2]:
PROJECT_ROOT = Path("D:/adaptive_federated_gnn")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "day10"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GRAPH_PATH = PROCESSED_DIR / "edge_graph.pt"
EDGE_TABLE_PATH = PROCESSED_DIR / "edge_table.csv"

print("Graph path:", GRAPH_PATH)
print("Edge table path:", EDGE_TABLE_PATH)

Graph path: D:\adaptive_federated_gnn\data\processed\edge_graph.pt
Edge table path: D:\adaptive_federated_gnn\data\processed\edge_table.csv


In [3]:
edge_graph = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

edge_table = pd.read_csv(EDGE_TABLE_PATH)

edge_index = edge_graph["edge_index"]
edge_features = edge_graph["edge_attr"]
edge_labels = edge_graph["edge_label"]

print("Number of nodes:", edge_graph["edge_index"].max().item() + 1)
print("Number of edges:", edge_index.shape[1])
print("Number of edge features:", edge_features.shape[1])

print("\nEdge table shape:", edge_table.shape)
print("\nLabel distribution:")
print(edge_labels.bincount())

Number of nodes: 4404
Number of edges: 286095
Number of edge features: 67

Edge table shape: (286095, 11)

Label distribution:
tensor([127291, 158804])


In [4]:
assert len(edge_table) == edge_index.shape[1]
assert len(edge_table) == len(edge_labels)

print("Edge table and graph are correctly aligned.")

Edge table and graph are correctly aligned.


In [5]:
all_edge_indices = np.arange(len(edge_labels))

labels_numpy = edge_labels.numpy()

train_indices, test_indices = train_test_split(
    all_edge_indices,
    test_size=0.30,
    random_state=SEED,
    stratify=labels_numpy
)

print("Training edges:", len(train_indices))
print("Testing edges:", len(test_indices))

Training edges: 200266
Testing edges: 85829


In [6]:
train_labels_numpy = labels_numpy[train_indices]

client_1_indices, remaining_indices = train_test_split(
    train_indices,
    test_size=2 / 3,
    random_state=SEED,
    stratify=train_labels_numpy
)

remaining_labels = labels_numpy[remaining_indices]

client_2_indices, client_3_indices = train_test_split(
    remaining_indices,
    test_size=0.50,
    random_state=SEED,
    stratify=remaining_labels
)

clients = {
    "client_1": np.array(client_1_indices),
    "client_2": np.array(client_2_indices),
    "client_3": np.array(client_3_indices)
}

print("Client split completed.")

Client split completed.


In [7]:
client_summary = []

for client_name, indices in clients.items():
    client_labels = labels_numpy[indices]

    benign_count = int((client_labels == 0).sum())
    attack_count = int((client_labels == 1).sum())

    client_summary.append({
        "client": client_name,
        "total_edges": len(indices),
        "benign_edges": benign_count,
        "attack_edges": attack_count,
        "attack_percentage": round(
            attack_count / len(indices) * 100,
            4
        )
    })

client_summary_df = pd.DataFrame(client_summary)

display(client_summary_df)

,client,total_edges,benign_edges,attack_edges,attack_percentage
0,client_1,66755,29701,37054,55.5075
1,client_2,66755,29701,37054,55.5075
2,client_3,66756,29701,37055,55.5081


In [8]:
client_summary_df.to_csv(
    OUTPUT_DIR / "client_distribution.csv",
    index=False
)

print("Saved:", OUTPUT_DIR / "client_distribution.csv")

Saved: D:\adaptive_federated_gnn\outputs\day10\client_distribution.csv


In [9]:
client_indices_df = pd.DataFrame({
    "client_1": pd.Series(clients["client_1"]),
    "client_2": pd.Series(clients["client_2"]),
    "client_3": pd.Series(clients["client_3"])
})

client_indices_df.to_csv(
    OUTPUT_DIR / "client_edge_indices.csv",
    index=False
)

print("Saved:", OUTPUT_DIR / "client_edge_indices.csv")

Saved: D:\adaptive_federated_gnn\outputs\day10\client_edge_indices.csv


Each client receives:

The same complete graph topology.
Its own local edge features.
Its own local labels.
Its own training edge indices.

In [10]:
client_graphs = {}

for client_name, indices in clients.items():
    indices_tensor = torch.tensor(
        indices,
        dtype=torch.long
    )

    client_graphs[client_name] = {
        "edge_index": edge_index,
        "edge_features": edge_features[indices_tensor],
        "edge_labels": edge_labels[indices_tensor],
        "local_edge_indices": indices_tensor
    }

    print(
        client_name,
        "->",
        len(indices),
        "local edges"
    )

client_1 -> 66755 local edges
client_2 -> 66755 local edges
client_3 -> 66756 local edges


In [11]:
for client_name, client_graph in client_graphs.items():
    labels = client_graph["edge_labels"]

    print(f"\n{client_name}")
    print("Total:", len(labels))
    print("Benign:", int((labels == 0).sum()))
    print("Attack:", int((labels == 1).sum()))


client_1
Total: 66755
Benign: 29701
Attack: 37054

client_2
Total: 66755
Benign: 29701
Attack: 37054

client_3
Total: 66756
Benign: 29701
Attack: 37055


In [12]:
CLIENT_GRAPH_PATH = OUTPUT_DIR / "federated_clients.pt"

torch.save(
    client_graphs,
    CLIENT_GRAPH_PATH
)

print("Saved:", CLIENT_GRAPH_PATH)

Saved: D:\adaptive_federated_gnn\outputs\day10\federated_clients.pt


In [13]:
loaded_clients = torch.load(
    CLIENT_GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

print("Loaded clients:", list(loaded_clients.keys()))

for client_name, client_graph in loaded_clients.items():
    print(
        client_name,
        "edges:",
        client_graph["edge_features"].shape[0],
        "features:",
        client_graph["edge_features"].shape[1]
    )

Loaded clients: ['client_1', 'client_2', 'client_3']
client_1 edges: 66755 features: 67
client_2 edges: 66755 features: 67
client_3 edges: 66756 features: 67


In [14]:
client_sets = {
    name: set(indices.tolist())
    for name, indices in clients.items()
}

overlap_12 = client_sets["client_1"].intersection(
    client_sets["client_2"]
)

overlap_13 = client_sets["client_1"].intersection(
    client_sets["client_3"]
)

overlap_23 = client_sets["client_2"].intersection(
    client_sets["client_3"]
)

print("Client 1 and Client 2 overlap:", len(overlap_12))
print("Client 1 and Client 3 overlap:", len(overlap_13))
print("Client 2 and Client 3 overlap:", len(overlap_23))

assert len(overlap_12) == 0
assert len(overlap_13) == 0
assert len(overlap_23) == 0

print("No client data overlap detected.")

Client 1 and Client 2 overlap: 0
Client 1 and Client 3 overlap: 0
Client 2 and Client 3 overlap: 0
No client data overlap detected.


In [15]:
combined_client_indices = (
    client_sets["client_1"]
    | client_sets["client_2"]
    | client_sets["client_3"]
)

assert len(combined_client_indices) == len(train_indices)

print("All training edges are assigned to clients.")

All training edges are assigned to clients.


In [16]:
federated_config = {
    "number_of_clients": 3,
    "random_seed": SEED,
    "total_edges": len(edge_labels),
    "training_edges": len(train_indices),
    "testing_edges": len(test_indices),
    "client_names": list(clients.keys()),
    "split_type": "stratified IID simulation",
    "graph_type": "directed IP flow graph",
    "learning_task": "edge-level binary classification"
}

torch.save(
    federated_config,
    OUTPUT_DIR / "federated_config.pt"
)

print("Saved federated configuration.")

Saved federated configuration.


In [17]:
print("=" * 60)
print("DAY 10 COMPLETED")
print("=" * 60)

print("Number of clients:", len(client_graphs))
print("Total graph edges:", len(edge_labels))
print("Training edges:", len(train_indices))
print("Testing edges:", len(test_indices))

for client_name, client_graph in client_graphs.items():
    labels = client_graph["edge_labels"]

    print(
        f"{client_name}: "
        f"{len(labels)} edges | "
        f"Benign: {(labels == 0).sum().item()} | "
        f"Attack: {(labels == 1).sum().item()}"
    )

print("\nFiles created:")
print("1.", OUTPUT_DIR / "client_distribution.csv")
print("2.", OUTPUT_DIR / "client_edge_indices.csv")
print("3.", OUTPUT_DIR / "federated_clients.pt")
print("4.", OUTPUT_DIR / "federated_config.pt")

DAY 10 COMPLETED
Number of clients: 3
Total graph edges: 286095
Training edges: 200266
Testing edges: 85829
client_1: 66755 edges | Benign: 29701 | Attack: 37054
client_2: 66755 edges | Benign: 29701 | Attack: 37054
client_3: 66756 edges | Benign: 29701 | Attack: 37055

Files created:
1. D:\adaptive_federated_gnn\outputs\day10\client_distribution.csv
2. D:\adaptive_federated_gnn\outputs\day10\client_edge_indices.csv
3. D:\adaptive_federated_gnn\outputs\day10\federated_clients.pt
4. D:\adaptive_federated_gnn\outputs\day10\federated_config.pt


Number of clients: 3
Total graph edges: 286095
Training edges: 200266
Testing edges: 85829

client_1: approximately 66755 edges
client_2: approximately 66755 edges
client_3: approximately 66756 edges

We have now simulated three distributed organizations that possess different subsets of network-flow data